In [1]:
import pandas as pd
import numpy as np

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split, GridSearchCV
from surprise import accuracy
from collections import defaultdict

In [2]:
# TCVファイルの読み込み
df_train = pd.read_csv('../000.data/train/train_C.tsv', sep="\t")
test = pd.read_csv('../000.data/test.tsv', sep="\t")

In [3]:
# 末尾が "_C" のテストデータだけを抽出
df_test = test[test["user_id"].str.endswith("_C")]

In [4]:
# 広告経由の購入のみ関連度3、それ以外はスコアをそのまま使用
def calculate_relevance(row):
    if row['event_type'] == 3 and row['ad'] == 1:
        return 3
    return row['event_type']

df_train['rating'] = df_train.apply(calculate_relevance, axis=1)

In [5]:
# Surprise用データセット形式に変換
reader = Reader(rating_scale=(0, 3))
data = Dataset.load_from_df(df_train[['user_id', 'product_id', 'rating']], reader)

In [6]:
# 学習データと検証データに分割
trainset, validationset = train_test_split(data, test_size=0.2)

In [7]:
# ハイパーパラメータの自動調整
param_grid = {
    'n_factors': [50, 100],
    'lr_all': [0.002, 0.005],
    'reg_all': [0.02, 0.1]
}
gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3)
gs.fit(data)

# 最適パラメータでモデル作成
print(f"Best RMSE Score: {gs.best_score['rmse']}")
print(f"Best Parameters: {gs.best_params['rmse']}")
model = SVD(**gs.best_params['rmse'])

Best RMSE Score: 0.11152922240194123
Best Parameters: {'n_factors': 50, 'lr_all': 0.005, 'reg_all': 0.1}


In [8]:
# nDCG計算
def calculate_ndcg(predictions, k=22):
    def dcg(scores):
        return sum([score / np.log2(idx + 2) for idx, score in enumerate(scores)])

    user_ranking = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_ranking[uid].append((iid, est, true_r))

    ndcg_values = []
    for uid, user_preds in user_ranking.items():
        user_preds.sort(key=lambda x: x[1], reverse=True)
        relevance_scores = [x[2] for x in user_preds[:k]]
        ideal_scores = sorted(relevance_scores, reverse=True)
        ndcg = dcg(relevance_scores) / dcg(ideal_scores) if dcg(ideal_scores) > 0 else 0
        ndcg_values.append(ndcg)

    return np.mean(ndcg_values)

In [9]:
# nDCGの評価を検証データで実施
model.fit(trainset)
predictions = model.test(validationset)
print(f"Validation nDCG Score: {calculate_ndcg(predictions):.4f}")

Validation nDCG Score: 0.9956


In [10]:
# 提出用データ作成
user_ids = df_test['user_id'].unique()
submission_df = pd.DataFrame(columns=['user_id', 'product_id', 'rank'])

for uid in user_ids:
    all_predictions = [model.predict(uid, iid) for iid in df_train['product_id'].unique()]
    all_predictions.sort(key=lambda x: x.est, reverse=True)
    top_k = all_predictions[:22]
    submission_df = pd.concat([submission_df, pd.DataFrame([[uid, pred.iid, rank + 1] for rank, pred in enumerate(top_k)], columns=['user_id', 'product_id', 'rank'])])

In [11]:
submission_df.to_csv('./predict_file/recommendation_result_C.tsv', sep="\t", index=False)